# Assignment 4: Diabetes Prediction using K-Nearest Neighbors

**Aim:** Apply the K-Nearest Neighbors algorithm to `diabetes.csv` and calculate the confusion matrix, accuracy, error rate, precision and recall.

**Dataset:** Download the CSV from the Kaggle assignment link. The notebook assumes the common `Outcome` target column; edit `TARGET_COLUMN` if needed.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report

DATA_PATH = "diabetes.csv"  # Place the downloaded CSV in the same folder
TARGET_COLUMN = "Outcome"   # Change if your target column has another name

## 2. Load and inspect dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())
display(df.isnull().sum())
display(df.describe().T)

## 3. Preprocess the data
Separate input features from the target. In the common Pima Indians diabetes dataset, zero values in Glucose, BloodPressure, SkinThickness, Insulin and BMI can represent missing measurements. The cell below replaces those zeros with NaN before median imputation. It does not modify legitimate zero values in Pregnancies or DiabetesPedigreeFunction.

In [ ]:
data = df.copy()
if TARGET_COLUMN not in data.columns:
    raise ValueError(f"Target '{TARGET_COLUMN}' not found. Available columns: {data.columns.tolist()}")

# Convert columns to numeric where possible
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_as_missing:
    if col in data.columns:
        data.loc[data[col] == 0, col] = np.nan

data = data.dropna(subset=[TARGET_COLUMN])
X = data.drop(columns=[TARGET_COLUMN])
y = data[TARGET_COLUMN].astype(int)

print("Target distribution:")
display(y.value_counts())

## 4. Split and scale data
Scaling is important for KNN because it uses distances. Imputation and scaling are placed in a pipeline so statistics are learned from training data only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

knn_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

## 5. Train KNN classifier

In [ ]:
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)
print("KNN model trained successfully.")

## 6. Evaluate model

In [ ]:
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Confusion Matrix:\\n", cm)
print("Accuracy:  {:.4f} ({:.2f}%)".format(accuracy, accuracy * 100))
print("Error Rate:{:.4f} ({:.2f}%)".format(error_rate, error_rate * 100))
print("Precision: {:.4f}".format(precision))
print("Recall:    {:.4f}".format(recall))
print("\\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Diabetes (0)", "Diabetes (1)"],
            yticklabels=["No Diabetes (0)", "Diabetes (1)"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("KNN Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. Conclusion
The KNN classifier was trained on the diabetes dataset after handling missing/invalid measurements and scaling the features. The confusion matrix and calculated accuracy, error rate, precision and recall describe its performance. Recall for class 1 indicates how many actual diabetes cases were correctly identified. These are educational model results, not a medical diagnosis.